In [1]:
import os
import sys
from pathlib import Path

LIB = Path.cwd().parent / "jobs"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from pyspark.sql import SparkSession, functions as F

SPARK_CONNECT_URL = os.environ.get(
    "SPARK_CONNECT_URL",
    "sc://192.168.2.180:15002",
)

spark = (
    SparkSession.builder
    .appName("HA Network Traffic Exploration")
    .remote(SPARK_CONNECT_URL)
    .getOrCreate()
)

LAKEHOUSE_ROOT = os.environ.get("LAKEHOUSE_ROOT", "s3a://lakehouse")
RAW_PATH = "s3a://raw/network_traffic/*/*.jsonl"
TABLE_PATH = f"{LAKEHOUSE_ROOT}/network_traffic/packets"

In [3]:
PACKET_SCHEMA = (
    "time double, src_ip string, dst_ip string, ip_proto string, "
    "length long, src_port long, dst_port long, "
    "dns_query string, tls_sni string, http_host string, http_uri string, "
    "protocol string, info string"
)

raw = (
    spark.read.schema(PACKET_SCHEMA).json(RAW_PATH)
    # The date the add-on already put in the object key rather than re-deriving
    # one from `time`.
    .withColumn("date", F.regexp_extract(F.input_file_name(),
                                          r"network_traffic/(\d{4}-\d{2}-\d{2})/", 1))
)

# Preview first so the notebook gives quick feedback without scanning all files.
raw.limit(10).show(truncate=60)

# Run this separately when you need the exact total; it scans every matching file.
# print("rows:", raw.count())

+-------------------+------------+------------+--------+------+--------+--------+---------+-------+---------+--------+--------+------------------------------------------------------------+----------+
|               time|      src_ip|      dst_ip|ip_proto|length|src_port|dst_port|dns_query|tls_sni|http_host|http_uri|protocol|                                                        info|      date|
+-------------------+------------+------------+--------+------+--------+--------+---------+-------+---------+--------+--------+------------------------------------------------------------+----------+
|1.787341379928299E9|172.30.33.16| 172.30.32.1|       6|  1188|   33306|    5432|     NULL|   NULL|     NULL|    NULL|   PGSQL|                                                          >Q|2026-08-21|
| 1.78734137992831E9|172.30.33.16| 172.30.32.1|       6|  1188|   33306|    5432|     NULL|   NULL|     NULL|    NULL|     TCP|[TCP Retransmission] 33306 → 5432 [PSH, ACK] Seq=1 Ack=1 ...|2026-08-21|


## Traffic overview

The raw rows are useful after grouping them into conversations, services, and time windows. The next cell adds a timestamp and private-network classification without collecting packet data to the notebook.

In [5]:
traffic = (
    raw.withColumn("timestamp", F.to_timestamp(F.from_unixtime("time")))
       .withColumn("hour", F.date_trunc("hour", "timestamp"))
       .withColumn(
           "src_private",
           F.col("src_ip").rlike(r"^(10\.|192\.168\.|172\.(1[6-9]|2[0-9]|3[0-1])\.)"),
       )
       .withColumn(
           "dst_private",
           F.col("dst_ip").rlike(r"^(10\.|192\.168\.|172\.(1[6-9]|2[0-9]|3[0-1])\.)"),
       )
)

traffic.select(
    F.min("timestamp").alias("first_packet"),
    F.max("timestamp").alias("last_packet"),
    F.count("*").alias("packets"),
    F.sum("length").alias("bytes"),
    F.countDistinct("src_ip").alias("unique_sources"),
    F.countDistinct("dst_ip").alias("unique_destinations"),
).withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2)).show(truncate=False)

print("Private-to-private / private-to-public direction counts")
traffic.groupBy(
    F.when(F.col("src_private") & F.col("dst_private"), "private -> private")
     .when(F.col("src_private") & ~F.col("dst_private"), "private -> public")
     .when(~F.col("src_private") & F.col("dst_private"), "public -> private")
     .otherwise("public -> public")
     .alias("direction")
).count().orderBy(F.col("count").desc()).show()

+-------------------+-------------------+-------+----------+--------------+-------------------+---------+
|first_packet       |last_packet        |packets|bytes     |unique_sources|unique_destinations|megabytes|
+-------------------+-------------------+-------+----------+--------------+-------------------+---------+
|2026-08-15 17:19:32|2026-08-21 22:27:58|3911216|7213378723|152           |156                |6879.21  |
+-------------------+-------------------+-------+----------+--------------+-------------------+---------+

Private-to-private / private-to-public direction counts
+------------------+-------+
|         direction|  count|
+------------------+-------+
|private -> private|3698654|
| private -> public| 104428|
| public -> private|  64915|
|  public -> public|  43219|
+------------------+-------+



## Main conversations and services

These rankings show where the traffic volume actually went. Bytes are usually more meaningful than packet counts for finding heavy users.

In [6]:
print("Top conversations")
(traffic.groupBy("src_ip", "dst_ip")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("bytes").desc())
 .show(25, truncate=False))

print("Top destination services")
(traffic.groupBy("dst_port", "protocol")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"),
      F.countDistinct("dst_ip").alias("destination_ips"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("bytes").desc())
 .show(25, truncate=False))

Top conversations
+---------------+-------------+-------+----------+---------+
|src_ip         |dst_ip       |packets|bytes     |megabytes|
+---------------+-------------+-------+----------+---------+
|172.30.32.1    |172.30.33.6  |131109 |6038200768|5758.48  |
|172.30.32.1    |172.30.33.5  |671389 |189132278 |180.37   |
|192.168.2.102  |172.30.33.19 |254628 |186213676 |177.59   |
|172.30.33.16   |172.30.32.1  |548605 |174268954 |166.2    |
|172.30.33.5    |172.30.32.1  |625363 |165089938 |157.44   |
|172.30.32.1    |172.30.33.16 |512411 |135045828 |128.79   |
|192.168.2.102  |192.168.2.180|128846 |94356793  |89.99    |
|172.30.32.2    |172.30.33.13 |5246   |55475864  |52.91    |
|172.30.32.1    |172.30.33.9  |43996  |21175496  |20.19    |
|172.30.32.2    |172.30.33.18 |57564  |17823160  |17.0     |
|172.30.33.19   |192.168.2.102|235372 |17006544  |16.22    |
|172.30.33.8    |172.30.32.1  |87775  |10961142  |10.45    |
|172.30.32.1    |172.30.33.8  |69090  |8709166   |8.31     |
|192.1

## Names exposed by the traffic

DNS and TLS SNI reveal requested names when the protocols expose them. Empty results are expected for traffic that contains no DNS or TLS metadata.

In [7]:
print("Top DNS queries")
(traffic.where(F.col("dns_query").isNotNull())
 .groupBy("dns_query")
 .agg(F.count("*").alias("queries"))
 .orderBy(F.col("queries").desc())
 .show(30, truncate=False))

print("Top TLS server names")
(traffic.where(F.col("tls_sni").isNotNull())
 .groupBy("tls_sni")
 .agg(F.count("*").alias("packets"), F.sum("length").alias("bytes"))
 .withColumn("megabytes", F.round(F.col("bytes") / 1048576, 2))
 .orderBy(F.col("packets").desc())
 .show(30, truncate=False))

Top DNS queries
+---------------------------------------------------------------------+-------+
|dns_query                                                            |queries|
+---------------------------------------------------------------------+-------+
|dp-eftc-prod.eftc.local                                              |16237  |
|_companion-link._tcp.local                                           |4721   |
|is005vs03178.dev.ilias.local                                         |4291   |
|256A7BCF-67F0-447E-8A7E-90ADE936CB8B._asquic._udp.local              |2981   |
|_googlezone._tcp.local                                               |870    |
|_googlecast._tcp.local                                               |841    |
|core-samba.local.hass.io                                             |728    |
|www.google.com                                                       |492    |
|chromecast-hd-1ae927d817117347b4aeb4942140c098._googlecast._tcp.local|465    |
|1ae927d8-1711-7347-b4ae